# Stage 5 ? Training dataset preparation (bounded inspection)

This notebook constructs a small number of examples through the Stage 2 loader, applies the Stage 3 configuration, validates/encodes labels with Stage 4, and inspects the reciter-group split manifest. It does **not** train, materialize, or save the full corpus. Run from the repository root after `python -m pip install -e ".[notebook]"`. Network access is used only for a few streamed rows and their short audio clips.

## 1. Source and reproducibility setup

In [1]:
import json
from pathlib import Path
import pandas as pd
from quran_dataset import DatasetConfig, DatasetLoader
from quran_audio import AudioProcessingConfig, analyze_audio
from quran_annotations import validate_record
from quran_training_data import (
    DatasetStatistics, InclusionPolicy, LabelCodec, assert_no_group_leakage,
    expected_random_segment_group_leakage, prepare_record,
)

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists())
dataset_config = DatasetConfig.from_file(ROOT / "configs/dataset.json")
loader = DatasetLoader(dataset_config)
audio_config = AudioProcessingConfig.from_file(str(ROOT / "configs/audio_preprocessing.json"))
training_config = json.loads((ROOT / "configs/training_data.json").read_text(encoding="utf-8"))["inclusion_policy"]
policy = InclusionPolicy(**training_config)
manifest = json.loads((ROOT / "configs/reciter_split_manifest.json").read_text(encoding="utf-8"))


## 2. Dataset and vocabulary facts

The upstream repository includes model/tokenizer code but no reproducible training data loop. Local implementation paths and checkpoint evidence are summarized in `docs/training_dataset_report.md`.

In [2]:
codec = LabelCodec()
{
    "dataset": dataset_config.dataset_id,
    "recitation_configurations": len(dataset_config.recitation_configurations),
    "phoneme_symbols": len(codec.phoneme_to_id),
    "phoneme_output_size_including_pad_blank": codec.phoneme_vocab_size,
    "output_level_sizes": codec.level_vocab_sizes,
    "padding_id": 0,
    "ctc_blank_id": 0,
    "ignored_batch_padding": -100,
}

{'dataset': 'obadx/muaalem-annotated-v3',
 'recitation_configurations': 27,
 'phoneme_symbols': 42,
 'phoneme_output_size_including_pad_blank': 43,
 'output_level_sizes': {'phonemes': 43,
  'hams_or_jahr': 3,
  'shidda_or_rakhawa': 4,
  'tafkheem_or_taqeeq': 4,
  'itbaq': 3,
  'safeer': 3,
  'qalqla': 3,
  'tikraar': 3,
  'tafashie': 3,
  'istitala': 3,
  'ghonna': 3},
 'padding_id': 0,
 'ctc_blank_id': 0,
 'ignored_batch_padding': -100}

The pinned `quran-transcript==0.6.1` inventory contains 42 ordinary symbols. The original checkpoint head contains 43 classes: `[PAD]`/CTC blank 0 plus those 42 symbols. The old documentation heading says ?43?, but the package and saved model vocabulary show no extra phoneme. The original phonetizer supports `remove_spaces=True`; source `phonemes` are retained below with a separate space-free target string because the tokenizer delimiter ID is outside the phoneme head.

## 3. Inclusion policy

In [3]:
json.loads((ROOT / "configs/training_data.json").read_text(encoding="utf-8"))["inclusion_policy"]

{'include_non_quran': True,
 'expected_sample_rate': 16000,
 'require_mono': True,
 'min_match_ratio': None,
 'min_duration_seconds': None,
 'max_duration_seconds': None,
 'on_missing_annotation': 'exclude',
 'on_invalid_annotation': 'exclude'}

The default retains valid non-Quran segments and null spans; they are not treated as invalid. Missing/invalid targets and undecodable audio become explicit exclusions with reason codes. No match-ratio or duration thresholds are enabled because their semantics/distributions have not been established.

## 4. Limited real examples via the Stage 2 loader

In [4]:
selected_configurations = ["moshaf_0.1", "moshaf_1.0", "moshaf_26.1"]
stats = DatasetStatistics()
examples = []
exclusions = []
source_for_leakage = []
for configuration in selected_configurations:
    for row in loader.iter_samples(configuration, split="train", limit=2):
        # Snapshot source annotations and confirm preparation is read-only.
        source_labels = {key: row.get(key) for key in ("phonemes", "sifat", "imlaey", "uthmani")}
        result = prepare_record(row, configuration, audio_config=audio_config, policy=policy,
                                codec=codec, split_manifest=manifest)
        assert source_labels == {key: row.get(key) for key in source_labels}
        stats.observe(row, result)
        if result.excluded:
            exclusions.append({"source_key": result.excluded.source_key,
                               "reasons": result.excluded.reasons})
            continue
        example = result.example
        assert example is not None
        assert codec.decode_phonemes(example.labels["phonemes"]) == example.phoneme_text_for_encoding
        assert len(example.phoneme_groups) == len(example.group_phoneme_ids)
        assert len(example.phoneme_groups) == len(example.labels["ghonna"])
        for level in codec.sifat_to_id:
            assert codec.decode_sifat(level, example.labels[level]) == [entry[level] for entry in row["sifat"]]
        examples.append((row, example))
        source_for_leakage.append((example.source_key, row, example.split))
print("included", len(examples), "excluded", len(exclusions), "reasons", exclusions)

included 6 excluded 0 reasons []


## 5. Prepared audio and labels

In [5]:
display(pd.DataFrame([
    {
        "source_key": ex.source_key,
        "split": ex.split,
        "has_quran": ex.metadata["has_quran"],
        "source_phonemes": ex.phoneme_text,
        "phoneme_target_length": len(ex.labels["phonemes"]),
        "sifat_groups": len(ex.phoneme_groups),
        "first_group": ex.phoneme_groups[0],
        "first_ghonna_id": ex.labels["ghonna"][0],
        "audio_rate": ex.audio.sampling_rate,
        "audio_channels": ex.audio.channels,
        "audio_duration": ex.audio.original.duration_seconds,
        "audio_operations": [(op.name, op.applied) for op in ex.audio.operations],
        "trace": {k: ex.metadata[k] for k in ("moshaf_id", "reciter_id", "segment_index")},
    }
    for _, ex in examples
]))

,source_key,split,has_quran,source_phonemes,phoneme_target_length,sifat_groups,first_group,first_ghonna_id,audio_rate,audio_channels,audio_duration,audio_operations,trace
0,moshaf_0.1:0.1:001.0000,train,False,ءَعُۥۥذُ بِللَااهِ مِنَ ششَيطَاانِ ررَجِۦۦۦۦم,41,19,ءَ,2,16000,1,6.06,"[(to_float32, False), (resample, False), (nois...","{'moshaf_id': '0.1', 'reciter_id': 0, 'segment..."
1,moshaf_0.1:0.1:001.0001,train,True,بِسمِ للَااهِ ررَحمَاانِ ررَحِۦۦۦۦم,32,15,بِ,2,16000,1,6.02,"[(to_float32, False), (resample, False), (nois...","{'moshaf_id': '0.1', 'reciter_id': 0, 'segment..."
2,moshaf_1.0:1.0:001.0000,train,False,ءَعُۥۥذُ بِللَااهِ مِنَ ششَيطَاانِ ررَجِۦۦم,39,19,ءَ,2,16000,1,4.90,"[(to_float32, False), (resample, False), (nois...","{'moshaf_id': '1.0', 'reciter_id': 1, 'segment..."
3,moshaf_1.0:1.0:001.0001,train,True,بِسمِ للَااهِ ررَحمَاانِ ررَحِۦۦم,30,15,بِ,2,16000,1,4.56,"[(to_float32, False), (resample, False), (nois...","{'moshaf_id': '1.0', 'reciter_id': 1, 'segment..."
4,moshaf_26.1:26.1:001.0000,train,False,ءَعُۥۥذُ بِللَااهِ مِنَ ششَيطَاانِ ررَجِۦۦۦۦم,41,19,ءَ,2,16000,1,4.71,"[(to_float32, False), (resample, False), (nois...","{'moshaf_id': '26.1', 'reciter_id': 26, 'segme..."
5,moshaf_26.1:26.1:001.0001,train,True,بِسمِ للَااهِ ررَحمَاانِ ررَحِۦۦۦۦم,32,15,بِ,2,16000,1,5.08,"[(to_float32, False), (resample, False), (nois...","{'moshaf_id': '26.1', 'reciter_id': 26, 'segme..."


The displayed `source_phonemes` remain verbatim. `phoneme_target_length` counts encoded symbols after upstream-compatible inter-word space removal. Each sifat output head contains one label per reference group; phoneme character count and group count are intentionally different.

## 6. Annotation and target checks

In [6]:
[
    {
        "source_key": ex.source_key,
        "annotation_errors": [issue.code for issue in validate_record(row).errors],
        "annotation_warnings": [issue.code for issue in validate_record(row).warnings],
        "phoneme_round_trip": codec.decode_phonemes(ex.labels["phonemes"]) == ex.phoneme_text_for_encoding,
        "phoneme_groups_equal_sifat": [entry["phonemes"] for entry in row["sifat"]] == list(ex.phoneme_groups),
        "sifat_level_lengths": {name: len(ex.labels[name]) for name in codec.sifat_to_id},
        "audio_measurements": analyze_audio(ex.audio).as_dict(),
    }
    for row, ex in examples
]

[{'source_key': 'moshaf_0.1:0.1:001.0000',
  'annotation_errors': [],
  'annotation_warnings': [],
  'phoneme_round_trip': True,
  'phoneme_groups_equal_sifat': True,
  'sifat_level_lengths': {'hams_or_jahr': 19,
   'shidda_or_rakhawa': 19,
   'tafkheem_or_taqeeq': 19,
   'itbaq': 19,
   'safeer': 19,
   'qalqla': 19,
   'tikraar': 19,
   'tafashie': 19,
   'istitala': 19,
   'ghonna': 19},
  'audio_measurements': {'sampling_rate': 16000,
   'channels': 1,
   'duration_seconds': 6.06,
   'samples': 96960,
   'minimum_amplitude': -0.53656005859375,
   'maximum_amplitude': 0.59124755859375,
   'peak_amplitude': 0.59124755859375,
   'peak_dbfs': -4.564612791178253,
   'rms_amplitude': 0.08607004504083239,
   'rms_dbfs': -21.302959395321416,
   'clipping_threshold': 0.99999,
   'clipped_sample_count': 0,
   'clipped_sample_fraction': 0.0,
   'finite': True,
   'leading_silence_seconds': 0.4525,
   'trailing_silence_seconds': 0.0,
   'silence_threshold_dbfs': -60.0}},
 {'source_key': 'mosha

## 7. Split strategy and leakage comparison

The committed split manifest assigns every reciter and all their moshaf sets to one split. This prevents reciter/moshaf leakage. Different reciters may recite the same surahs, so this is speaker-independent evaluation, not text-exclusive evaluation.

In [7]:
split_counts = manifest["actual_source_row_counts"]
reciter_row_counts = {str(group["reciter_id"]): group["row_count"] for group in manifest["groups"]}
random_leakage = expected_random_segment_group_leakage(reciter_row_counts)
assert_no_group_leakage([
    (str(group["reciter_id"]), {"reciter_id": group["reciter_id"]}, group["split"])
    for group in manifest["groups"]
])
{
    "dataset_revision": manifest["dataset_revision"],
    "group_counts": {split: sum(g["split"] == split for g in manifest["groups"])
                     for split in ("train", "validation", "test")},
    "moshaf_set_counts": {split: sum(v == split for v in manifest["moshaf_id_to_split"].values())
                          for split in ("train", "validation", "test")},
    "source_row_counts_before_quality_filtering": split_counts,
    "expected_reciter_overlap_under_random_segment_split": random_leakage,
}

{'dataset_revision': '8e3fe50ec5efcc4beab054acdbd0178bc690c246',
 'group_counts': {'train': 16, 'validation': 3, 'test': 3},
 'moshaf_set_counts': {'train': 21, 'validation': 3, 'test': 3},
 'source_row_counts_before_quality_filtering': {'train': 228941,
  'validation': 29125,
  'test': 28471},
 'expected_reciter_overlap_under_random_segment_split': {'assumption': 'independent per-row random assignment with the requested probabilities',
  'expected_groups_present_in_each_split': {'train': 22.0,
   'validation': 22.0,
   'test': 22.0},
  'expected_groups_present_in_multiple_splits': 22.0,
  'group_count': 22}}

## 8. Bounded-sample statistics and limitations

In [8]:
stats.summary(rare_count=5)

{'records': {'total_seen': 6,
  'included': 6,
  'excluded': 0,
  'exclusion_reasons': {}},
 'quran_flags': {'non_quran': 3, 'quran': 3},
 'unique_reciters': 3,
 'unique_moshaf_sets': 3,
 'audio_duration_seconds': {'count': 6,
  'min': 4.56,
  'median': 4.99,
  'p90': 6.039999999999999,
  'max': 6.06},
 'phoneme_sequence_length': {'count': 6,
  'min': 30.0,
  'median': 35.5,
  'p90': 41.0,
  'max': 41.0},
 'sifat_group_count': {'count': 6,
  'min': 15.0,
  'median': 17.0,
  'p90': 19.0,
  'max': 19.0},
 'phoneme_frequency': {'ء': 3,
  'ا': 24,
  'ب': 6,
  'ج': 3,
  'ح': 6,
  'ذ': 3,
  'ر': 18,
  'س': 3,
  'ش': 6,
  'ط': 3,
  'ع': 3,
  'ل': 12,
  'م': 15,
  'ن': 9,
  'ه': 6,
  'ي': 3,
  'َ': 30,
  'ُ': 6,
  'ِ': 30,
  'ۥ': 6,
  'ۦ': 20},
 'rare_phonemes_at_or_below_threshold': {'ء': 3,
  'ج': 3,
  'ذ': 3,
  'س': 3,
  'ط': 3,
  'ع': 3,
  'ي': 3},
 'sifat_category_frequency': {'ghonna': {'maghnoon': 24, 'not_maghnoon': 78},
  'hams_or_jahr': {'hams': 18, 'jahr': 84},
  'istitala': {'not_m

These distributions cover only the six bounded rows loaded above. The split manifest itself uses authoritative per-configuration row counts and metadata, so its group sizes cover 286,537 source rows. Corpus-wide usable/excluded counts and label frequencies require scanning all annotation records; audio was not downloaded for such a scan. `DatasetStatistics` can accumulate those statistics during a future streaming preparation pass without retaining waveforms.

## 9. Findings and conclusions

- The reference target contains 42 symbols plus PAD/blank; no 43rd ordinary symbol was missing.
- Unsupported symbols are errors, not mapped to an invented target ID.
- Space-free IDs match the original model tokenizer; original annotation strings remain attached to each prepared example.
- Ten independent sifat target levels use the original category order and pad ID; group counts are preserved.
- The split manifest is reproducible at dataset revision `8e3fe50ec5efcc4beab054acdbd0178bc690c246`, grouped by reciter.
- No model training, full dataset materialization, or label modification occurs in this notebook.